# 07 — Headless

Every chapter so far had a human in a cell pressing go. This one takes the human away, then
puts one back exactly where it matters.

> **You'll learn**
> - Register a **trigger** so the control plane owns the schedule, not your notebook
> - Prove a run happened with nothing attached to it, from the run id alone
> - Gate a destructive remediation behind `app.pause()` — and make it **fail closed**

In [1]:
import sys, json, time, requests
sys.path.insert(0, "../lib")
sys.path.insert(0, "../node")
import dag

SERVER = "http://localhost:8080"
S = requests.Session()

def get(path, **kw):  return S.get(f"{SERVER}{path}", timeout=30, **kw).json()
def post(path, body=None, timeout=600): return S.post(f"{SERVER}{path}", json=body or {}, timeout=timeout).json()

`r07_sweep` carries `@on_schedule` under `@router.reasoner`. Starting the node registered the
trigger; no provisioning call was made.

In [2]:
from rungs.r07 import SWEEP_CRON, DESTRUCTIVE
print("declared cron:", SWEEP_CRON, "(daily — a minute cron you forget about is a bill)")
print("destructive verbs:", ", ".join(DESTRUCTIVE[:8]), "...")

rows = [t for t in get("/api/v1/triggers")["triggers"]
        if t["target_node_id"] == "blast-radius" and t["target_reasoner"] == "r07_sweep"]
for t in rows:
    print(f'  {t["id"]}  {t["source_name"]}  {t["config"]["expression"]:12} managed_by={t["managed_by"]}  enabled={t["enabled"]}')

declared cron: 0 4 * * * (daily — a minute cron you forget about is a bill)
destructive verbs: roll back, rollback, revert, restart, reboot, failover, fail over, delete ...
  exec_20260820_121507_69d30rnz  cron  0 4 * * *    managed_by=code  enabled=False


`managed_by: "code"` with a source line is the tell: the control plane knows which file
declared this. Rows created through the API say `"ui"` instead.

Daily-at-04:00 is too slow to watch. Add a one-minute cron through the API — a `"ui"` row, so
it can be deleted afterwards — and wait for it to fire on its own.

In [3]:
tmp = post("/api/v1/triggers", {
    "source_name": "cron",
    "target_node_id": "blast-radius",
    "target_reasoner": "r07_sweep",
    "config": {"expression": "* * * * *", "timezone": "UTC"},
    "enabled": True,
})
TMP_ID = tmp["id"]
print("temporary trigger:", TMP_ID, "managed_by =", tmp["managed_by"])

temporary trigger: exec_20260820_123213_dfdlu70d managed_by = ui


Now nothing happens in this notebook. The next cell only *watches*.

In [4]:
deadline = time.time() + 150
events = []
while time.time() < deadline:
    events = get(f"/api/v1/triggers/{TMP_ID}/events").get("events", [])
    if events and events[0].get("dispatched_workflow_id"):
        break
    time.sleep(5)

ev = events[0]
print("fired_at     :", ev["raw_payload"]["fired_at"])
print("event_type   :", ev["event_type"])
print("idempotency  :", ev["idempotency_key"])
print("status       :", ev["status"])
print("run id       :", ev["dispatched_workflow_id"])

fired_at     : 2026-08-20T16:33:00Z
event_type   : tick
idempotency  : * * * * *@2026-08-20T16:33Z
status       : dispatched
run id       : wf_20260820_123300_h3dtw4en


The run id starts `wf_`, not `run_`. Direct calls get `run_`; anything the control plane
dispatched from an event gets `wf_`. That prefix is the proof, and it costs nothing to check.

In [5]:
WF = ev["dispatched_workflow_id"]
print("headless:", WF.startswith("wf_"))

# let it finish before we draw it -- the event beat the work by a whole diagnosis
for _ in range(90):
    run = get(f"/api/v1/agentic/run/{WF}")["data"]
    if all(e["status"] not in ("running", "queued") for e in run["executions"]):
        break
    time.sleep(5)

dag.render(WF, title=f"{WF} — nobody called this")

headless: True


**wf_20260820_123300_h3dtw4en — nobody called this** — 10 executions · depth 4 · max fan-out 6 · 1 agent(s)

```mermaid
flowchart TD
  n0["r07_sweep<br/><small>✓ succeeded · 48.1s</small>"]
  n1["r07_triage<br/><small>✓ succeeded · 47.9s</small>"]
  n2["r06_diagnose<br/><small>✓ succeeded · 42.3s</small>"]
  n3["r06_choose_lenses<br/><small>✓ succeeded · 10.0s</small>"]
  n4["r06_apply_lens<br/><small>✓ succeeded · 5.5s</small>"]
  n5["r06_apply_lens<br/><small>✓ succeeded · 6.1s</small>"]
  n6["r06_apply_lens<br/><small>✓ succeeded · 6.5s</small>"]
  n7["r06_apply_lens<br/><small>✓ succeeded · 7.0s</small>"]
  n8["r06_apply_lens<br/><small>✓ succeeded · 16.0s</small>"]
  n9["r07_classify_remediation<br/><small>✓ succeeded · 5.1s</small>"]
  n0 --> n1
  n1 --> n2
  n2 --> n3
  n2 --> n4
  n2 --> n5
  n2 --> n6
  n2 --> n7
  n2 --> n8
  n1 --> n9
  class n0,n1,n2,n3,n4,n5,n6,n7,n8,n9 ok;
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

The sweep diagnosed an incident and then hit the gate. `triage` classifies its own proposed
remediation before doing anything with it.

In [6]:
run = get(f"/api/v1/agentic/run/{WF}")["data"]
for e in sorted(run["executions"], key=lambda x: x["started_at"]):
    print(f'  {e["reasoner_id"]:26} {e["status"]:10} {e.get("duration_ms", 0)/1000:6.1f}s')
print()
for n in run.get("notes", []):
    print("note:", n["tags"], n["message"])

  r07_sweep                  succeeded    48.1s
  r07_triage                 succeeded    47.9s
  r06_diagnose               succeeded    42.3s
  r06_choose_lenses          succeeded    10.0s
  r06_apply_lens             succeeded     5.5s
  r06_apply_lens             succeeded     6.1s
  r06_apply_lens             succeeded     6.5s
  r06_apply_lens             succeeded     7.0s
  r06_apply_lens             succeeded    16.0s
  r07_classify_remediation   succeeded     5.1s

note: ['approval', 'gate'] destructive remediation proposed for inc-011: Synchronize the clock on node eu-c1-n07 by restarting chronyd or forcing a time sync, then verify the node clock offset metric returns to zero.


Here is the gate itself, run directly so we can read what came back. inc-006's fix is *revert
the deploy* — not something an agent gets to do on its own.

In [7]:
res = post("/api/v1/execute/blast-radius.r07_triage",
           {"input": {"incident_id": "inc-006", "auto_approve_timeout": 30}})
r = res["result"]
print("root cause :", r["root_cause"][:150])
print("destructive:", r["destructive"])
print("approval   :", r["approval"])
print("action     :", r["action"][:160])
print()
print("gate said  :", r["feedback"][:300])

root cause : Deployment dep-8814 at 04:10 changed the Redis cache key namespace from pcat:v7 to pcat:v8, invalidating all cached entries and causing a catastrophic
destructive: True
approval   : gate_unavailable
action     : HELD — gate unavailable, not executed: Roll back deployment dep-8814 to restore the cache key namespace to pcat:v7.

gate said  : Approval request failed (400): {"error":"invalid_callback_url","message":"callback_url rejected: webhook url must not target private/internal address 127.0.0.1"}


**An honest gap.** `app.pause()` never reached a human here. This control plane was started
without `AGENTFIELD_WEBHOOK_ALLOWED_HOSTS`, so its SSRF guard rejects the agent's own callback
address, and the approval request 400s before anyone is asked.

That is the right failure. The gate **fails closed**: no approval means the destructive action
is held, not run. An agent that reads a broken gate as consent is worse than one with no gate
at all.

In [8]:
import inspect
from rungs import r07
src = inspect.getsource(r07.triage)
print(src[src.index("    try:"):src.index("    feedback =")])

    try:
        res = await router.app.pause(
            approval_request_id=f"remediate-{incident_id}",
            approval_request_url=f"http://127.0.0.1:8080/review/{incident_id}",
            expires_in_hours=1,
            timeout=auto_approve_timeout,
        )
    except Exception as exc:  # noqa: BLE001
        # The control plane refuses a callback to a private address unless it was
        # started with AGENTFIELD_WEBHOOK_ALLOWED_HOSTS=localhost,127.0.0.1. Fail
        # CLOSED: no approval means the destructive action does not run. An agent
        # that treats a broken gate as consent is worse than one with no gate.
        return TriageResult(
            incident_id=incident_id,
            root_cause=diagnosis.root_cause,
            action=f"HELD — gate unavailable, not executed: {plan.action}",
            destructive=True,
            approval="gate_unavailable",
            feedback=str(exc)[:400],
        )



To run the gate for real, start the control plane with the guard configured and point the
agent's callback at `127.0.0.1` — never `localhost`, which the Go control plane resolves to
`::1` while uvicorn binds IPv4, giving you a control plane that says *approved* and an agent
that hangs forever. See `docs/approvals-triggers-memory.md`.

In [9]:
print('''
  AGENTFIELD_WEBHOOK_ALLOWED_HOSTS=localhost,127.0.0.1 af server
  AGENT_CALLBACK_URL=http://127.0.0.1:8002 python node/main.py

  curl -X POST $SERVER/api/v1/executions/$EXEC/approval-response \\
    -d '{"decision":"approved","response":{"feedback":"reviewed"}}'
                                ^^^^^^^^ nested. a top-level "feedback" is dropped.
'''.strip())

AGENTFIELD_WEBHOOK_ALLOWED_HOSTS=localhost,127.0.0.1 af server
  AGENT_CALLBACK_URL=http://127.0.0.1:8002 python node/main.py

  curl -X POST $SERVER/api/v1/executions/$EXEC/approval-response \
    -d '{"decision":"approved","response":{"feedback":"reviewed"}}'
                                ^^^^^^^^ nested. a top-level "feedback" is dropped.


## Teardown

A cron trigger keeps firing after its agent dies, and a code-managed row **cannot be deleted**
while its reasoner still registers. `pause` is the only off switch. Never leave a chapter
running on someone's laptop.

In [10]:
print(S.delete(f"{SERVER}/api/v1/triggers/{TMP_ID}", timeout=30).json())   # ui row: deletable

for t in rows:                                                             # code row: pause only
    print(t["id"], post(f'/api/v1/triggers/{t["id"]}/pause'))

for t in get("/api/v1/triggers")["triggers"]:
    if t["target_node_id"] == "blast-radius":
        print(f'  {t["id"]}  enabled={t["enabled"]}  override={t.get("manual_override_enabled")}')

{'status': 'deleted'}
exec_20260820_121507_69d30rnz {'status': 'paused'}
  exec_20260820_121507_69d30rnz  enabled=False  override=True


## What you learned

- A **trigger** moves ownership of "when" from your notebook to the control plane; the `wf_`
  run-id prefix is free proof that nothing was attached to the run.
- `app.pause()` puts a human inside an execution — and the gate must **fail closed**, because
  a gate that errors open is a liability, not a safeguard.
- Code-managed triggers **outlive your code**. `pause` is the off switch; plan the teardown
  before you write the decorator.

**Next:** nothing — this is the last rung. What you just ran is what `pr-af`, `swe-af` and
`sec-af` are: this same rung, on a real repository, triggered by a real webhook, gated on a
real person.